<a href="https://colab.research.google.com/github/flathfk/Bootcamp_Study/blob/main/260311.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌽 옥수수 선물 거래 시뮬레이터 최종 설계

## 1단계 - 시나리오
- 서비스명: 농산물 선물 거래 시뮬레이터
- 타겟: MZ세대 투자 입문자
- 핵심 시나리오: 가상 시드머니로 농산물 선물 거래 체험
- 기획 배경: 경제방송 PD 동생 + 전업 투자자 친구 리서치

## 2단계 - 알고리즘
- 초기 시드머니: 1,000만원
- 레버리지: 2x / 5x / 10x
- 가격 변동: 3초마다 ±3% 랜덤
- 수익 계산: 마진 × 레버리지 × 가격변동률
- 청산 조건: 손실이 마진 100% 도달 시 강제 청산

## 3단계 - 톤 & 문구
- 톤: 토스 느낌 (친근하고 캐주얼)
- 수익: 빨강 / 손실: 파랑 (한국 주식 스타일)
- 버튼: "매수할게요" / "청산할게요" / "다시 시작하기"

## 4단계 - UI 레이아웃
- 헤더: 잔고 / 수익률 표시
- 좌측 패널: 종목 선택, 롱/숏, 레버리지, 매수 버튼
- 우측 패널: 거래 내역 테이블
- 하단: 4개 종목 실시간 가격 현황

## 5단계 - 데이터 구조

### database.json
{
  "balance": 10000000,
  "trades": []
}

### data.js 종목 데이터
const ASSETS = [
  { id: "corn",    name: "옥수수", emoji: "🌽", price: 480  },
  { id: "wheat",   name: "밀",    emoji: "🌾", price: 620  },
  { id: "cattle",  name: "생우",  emoji: "🐄", price: 185  },
  { id: "soybean", name: "대두",  emoji: "🫘", price: 1420 }
]

- 🌽 옥수수 → 선물거래 기원, 스토리텔링 핵심
- 🌾 밀 → 우크라이나 전쟁, 뉴스에서 많이 나온 종목
- 🐄 생우 → 축산물 대표
- 🫘 대두 → 미국 농업 대표, 옥수수랑 단짝 종목

## 6~7단계 - 핵심 JS 로직
- setInterval → 3초마다 가격 랜덤 변동
- fetch POST → 거래 시 database.json에 저장
- fetch GET → 페이지 로드 시 거래 내역 불러오기
- DOM 조작 → 잔고/거래내역 테이블 실시간 업데이트

## 8단계 - 배포
- Express server.js → GET/POST API + 정적 파일 서빙
- Cloudflare Tunnel → 외부 공개 URL 생성
- GitHub → 전체 소스 업로드

## 파일 구조
```
/content/
├── server.js
├── database.json
├── templates/
│   └── index.html
└── static/
    └── data.js
```



---



In [14]:
!pkill -f "node /content/server.js"
!nohup node /content/server.js > /content/server.log 2>&1 &
!tail -n 5 /content/server.log



---



In [1]:
!mkdir -p /content/templates /content/static
!npm init -y
!npm install express

In [15]:
%%writefile /content/database.json

{
  "balance": 10000000,
  "trades": []
}

Overwriting /content/database.json


In [16]:
%%writefile /content/static/data.js

const ASSETS = [
  { id: "corn",    name: "옥수수", emoji: "🌽", price: 480  },
  { id: "wheat",   name: "밀",    emoji: "🌾", price: 620  },
  { id: "cattle",  name: "생우",  emoji: "🐄", price: 185  },
  { id: "soybean", name: "대두",  emoji: "🫘", price: 1420 }
];

Overwriting /content/static/data.js


In [17]:
%%writefile /content/server.js

const express = require("express");
const fs = require("fs");
const path = require("path");
const app = express();

app.use(express.json());
app.use(express.static(path.join(__dirname, "static")));

// 메인 페이지
app.get("/", (req, res) => {
  res.sendFile(path.join(__dirname, "templates", "index.html"));
});

// 거래 내역 + 잔고 불러오기
app.get("/api/data", (req, res) => {
  const data = JSON.parse(fs.readFileSync("/content/database.json", "utf8"));
  res.json(data);
});

// 거래 저장
app.post("/api/trade", (req, res) => {
  const data = JSON.parse(fs.readFileSync("/content/database.json", "utf8"));
  const trade = req.body;
  data.trades.push(trade);
  data.balance = trade.balanceAfter;
  fs.writeFileSync("/content/database.json", JSON.stringify(data, null, 2));
  res.json({ success: true });
});

// 잔고 초기화
app.post("/api/reset", (req, res) => {
  const init = { balance: 10000000, trades: [] };
  fs.writeFileSync("/content/database.json", JSON.stringify(init, null, 2));
  res.json({ success: true });
});

app.listen(3000, () => console.log("서버 실행 중 - port 3000"));

Overwriting /content/server.js


In [18]:
%%writefile /content/templates/index.html

<!DOCTYPE html>
<html lang="ko">
<head>
  <meta charset="UTF-8">
  <title>🌽 농산물 선물 거래 시뮬레이터</title>
  <style>
    :root {
      --primary: #0064FF;
      --profit: #e74c3c;
      --loss: #3498db;
      --bg: #f8f9fa;
      --card: #ffffff;
      --border: #e5e7eb;
      --text: #111827;
      --muted: #6b7280;
    }

    * { box-sizing: border-box; margin: 0; padding: 0; }
    body { font-family: 'Apple SD Gothic Neo', sans-serif; background: var(--bg); color: var(--text); }

    /* 헤더 */
    header {
      background: var(--primary);
      color: white;
      padding: 20px 40px;
      display: flex;
      justify-content: space-between;
      align-items: center;
    }
    header h1 { font-size: 20px; font-weight: 700; }
    .balance-box { text-align: right; }
    .balance-box .label { font-size: 12px; opacity: 0.8; }
    .balance-box .amount { font-size: 24px; font-weight: 700; }

    /* 메인 그리드 */
    .main-grid {
      display: grid;
      grid-template-columns: 340px 1fr;
      grid-template-rows: auto auto;
      gap: 20px;
      padding: 24px 40px;
      max-width: 1200px;
      margin: 0 auto;
    }

    /* 카드 공통 */
    .card {
      background: var(--card);
      border: 1px solid var(--border);
      border-radius: 16px;
      padding: 24px;
    }
    .card h2 { font-size: 15px; font-weight: 700; margin-bottom: 16px; color: var(--muted); }

    /* 거래 패널 */
    .trade-panel { grid-row: 1 / 3; }

    select, .leverage-btns button {
      width: 100%;
      padding: 12px;
      border: 1px solid var(--border);
      border-radius: 10px;
      font-size: 14px;
      margin-bottom: 12px;
      cursor: pointer;
    }

    .direction-btns { display: grid; grid-template-columns: 1fr 1fr; gap: 8px; margin-bottom: 12px; }
    .btn-long  { background: #fff0f0; color: var(--profit); border: 2px solid var(--profit); border-radius: 10px; padding: 12px; font-weight: 700; cursor: pointer; }
    .btn-short { background: #eff6ff; color: var(--loss);   border: 2px solid var(--loss);   border-radius: 10px; padding: 12px; font-weight: 700; cursor: pointer; }
    .btn-long.active  { background: var(--profit); color: white; }
    .btn-short.active { background: var(--loss);   color: white; }

    .leverage-btns { display: grid; grid-template-columns: 1fr 1fr 1fr; gap: 8px; margin-bottom: 12px; }
    .leverage-btns button { margin: 0; background: #f3f4f6; border: 2px solid var(--border); font-weight: 700; }
    .leverage-btns button.active { background: var(--primary); color: white; border-color: var(--primary); }

    .margin-input input {
      width: 100%;
      padding: 12px;
      border: 1px solid var(--border);
      border-radius: 10px;
      font-size: 14px;
      margin-bottom: 12px;
    }

    .btn-buy {
      width: 100%;
      padding: 14px;
      background: var(--primary);
      color: white;
      border: none;
      border-radius: 10px;
      font-size: 16px;
      font-weight: 700;
      cursor: pointer;
      margin-bottom: 8px;
    }
    .btn-reset {
      width: 100%;
      padding: 10px;
      background: #f3f4f6;
      color: var(--muted);
      border: none;
      border-radius: 10px;
      font-size: 13px;
      cursor: pointer;
    }

    /* 거래 내역 테이블 */
    .history-panel { overflow-x: auto; }
    table { width: 100%; border-collapse: collapse; font-size: 13px; }
    th { background: #f9fafb; padding: 10px 12px; color: var(--muted); font-weight: 600; border-bottom: 1px solid var(--border); text-align: left; }
    td { padding: 10px 12px; border-bottom: 1px solid var(--border); }
    .profit { color: var(--profit); font-weight: 700; }
    .loss   { color: var(--loss);   font-weight: 700; }

    /* 가격 현황 */
    .price-grid {
      grid-column: 1 / 3;
      display: grid;
      grid-template-columns: repeat(4, 1fr);
      gap: 16px;
    }
    .price-card {
      background: var(--card);
      border: 1px solid var(--border);
      border-radius: 16px;
      padding: 20px;
      text-align: center;
    }
    .price-card .emoji { font-size: 32px; margin-bottom: 8px; }
    .price-card .name  { font-size: 13px; color: var(--muted); margin-bottom: 4px; }
    .price-card .price { font-size: 22px; font-weight: 700; }
    .price-card .change { font-size: 12px; margin-top: 4px; }
  </style>
</head>
<body>

<header>
  <h1>🌽 농산물 선물 거래 시뮬레이터</h1>
  <div class="balance-box">
    <div class="label">내 잔고</div>
    <div class="amount" id="balance">10,000,000원</div>
  </div>
</header>

<div class="main-grid">

  <!-- 거래 패널 -->
  <div class="card trade-panel">
    <h2>거래하기</h2>

    <select id="asset-select">
      <option value="corn">🌽 옥수수</option>
      <option value="wheat">🌾 밀</option>
      <option value="cattle">🐄 생우</option>
      <option value="soybean">🫘 대두</option>
    </select>

    <div class="direction-btns">
      <button class="btn-long active"  id="btn-long"  onclick="setDirection('long')">▲ 롱 (상승 베팅)</button>
      <button class="btn-short"        id="btn-short" onclick="setDirection('short')">▼ 숏 (하락 베팅)</button>
    </div>

    <div class="leverage-btns">
      <button class="active" onclick="setLeverage(2,  this)">2x</button>
      <button               onclick="setLeverage(5,  this)">5x</button>
      <button               onclick="setLeverage(10, this)">10x</button>
    </div>

    <div class="margin-input">
      <input type="number" id="margin-input" placeholder="마진 입력 (원)" min="10000">
    </div>

    <button class="btn-buy"   onclick="executeTrade()">매수할게요</button>
    <button class="btn-reset" onclick="resetBalance()">다시 시작하기</button>
  </div>

  <!-- 거래 내역 -->
  <div class="card history-panel">
    <h2>거래 내역</h2>
    <table>
      <thead>
        <tr>
          <th>종목</th>
          <th>방향</th>
          <th>레버리지</th>
          <th>마진</th>
          <th>수익/손실</th>
          <th>잔고</th>
          <th>시간</th>
        </tr>
      </thead>
      <tbody id="trade-tbody"></tbody>
    </table>
  </div>

  <!-- 가격 현황 -->
  <div class="price-grid">
    <div class="price-card" id="price-corn">
      <div class="emoji">🌽</div>
      <div class="name">옥수수</div>
      <div class="price" id="p-corn">480</div>
      <div class="change" id="c-corn">-</div>
    </div>
    <div class="price-card" id="price-wheat">
      <div class="emoji">🌾</div>
      <div class="name">밀</div>
      <div class="price" id="p-wheat">620</div>
      <div class="change" id="c-wheat">-</div>
    </div>
    <div class="price-card" id="price-cattle">
      <div class="emoji">🐄</div>
      <div class="name">생우</div>
      <div class="price" id="p-cattle">185</div>
      <div class="change" id="c-cattle">-</div>
    </div>
    <div class="price-card" id="price-soybean">
      <div class="emoji">🫘</div>
      <div class="name">대두</div>
      <div class="price" id="p-soybean">1420</div>
      <div class="change" id="c-soybean">-</div>
    </div>
  </div>

</div>

<script src="/data.js"></script>
<script>
  // 상태 변수
  let balance = 10000000;
  let direction = "long";
  let leverage = 2;
  let prices = { corn: 480, wheat: 620, cattle: 185, soybean: 1420 };

  // 초기 잔고 불러오기
  fetch("/api/data")
    .then(r => r.json())
    .then(data => {
      balance = data.balance;
      updateBalance();
      renderTrades(data.trades);
    });

  // 잔고 표시
  function updateBalance() {
    document.getElementById("balance").textContent =
      balance.toLocaleString() + "원";
  }

  // 방향 선택
  function setDirection(dir) {
    direction = dir;
    document.getElementById("btn-long").classList.toggle("active",  dir === "long");
    document.getElementById("btn-short").classList.toggle("active", dir === "short");
  }

  // 레버리지 선택
  function setLeverage(val, el) {
    leverage = val;
    document.querySelectorAll(".leverage-btns button")
      .forEach(b => b.classList.remove("active"));
    el.classList.add("active");
  }

  // 가격 변동 (3초마다 ±3% 랜덤)
  setInterval(() => {
    ASSETS.forEach(asset => {
      const change = (Math.random() * 6 - 3) / 100; // -3% ~ +3%
      const prev = prices[asset.id];
      prices[asset.id] = Math.round(prev * (1 + change) * 100) / 100;

      const changeRate = ((prices[asset.id] - prev) / prev * 100).toFixed(2);
      document.getElementById("p-" + asset.id).textContent = prices[asset.id];

      const changeEl = document.getElementById("c-" + asset.id);
      changeEl.textContent = (changeRate > 0 ? "▲ " : "▼ ") + Math.abs(changeRate) + "%";
      changeEl.style.color = changeRate > 0 ? "var(--profit)" : "var(--loss)";
    });
  }, 3000);

  // 매수 실행
  function executeTrade() {
    const margin = parseInt(document.getElementById("margin-input").value);
    if (!margin || margin <= 0) { alert("마진을 입력해

Overwriting /content/templates/index.html


In [29]:
!nohup node /content/server.js > /content/server.log 2>&1 &
!tail -n 5 /content/server.log

In [30]:
!npm install -g cloudflared
!nohup cloudflared tunnel --url http://localhost:3000 > /content/tunnel.log 2>&1 &

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇
added 1 package in 2s
⠇

In [31]:
!grep -o "https://[a-zA-Z0-9.-]*\.trycloudflare.com" /content/tunnel.log | tail -n 1

https://seventh-pursue-dogs-budapest.trycloudflare.com
